# Phase 3 — The Change-Detection ("Diff") Layer

**How to use this:** copy each cell below into your existing notebook, right after the
"What You've Built So Far" section (before "Next Steps"). Run them in order.

This is the layer that turns your RAG engine from a Q&A chatbot into an actual **compliance
tool**: it explicitly tracks which policy version is current, detects what changed between
versions, and forces every answer to use the current version — while still being able to
explain what changed from the old one.

## Step 1 — Tag documents with version metadata

Right now `SAMPLE_DOCS` treats every document as equally "current." We need to tell the
system explicitly which documents are old versions and which are current, and which
policy family they belong to.

In [ ]:
# Add version metadata to the sample docs
# In a real system this would come from document dates / version numbers in the source PDFs

DOC_METADATA = {
    "policy_prior_auth_v1": {
        "policy_family": "prior_auth_outpatient_pt",
        "version": 1,
        "effective_date": "2026-01-01",
        "is_current": False,
    },
    "policy_prior_auth_v2": {
        "policy_family": "prior_auth_outpatient_pt",
        "version": 2,
        "effective_date": "2026-07-01",
        "is_current": True,
    },
    "policy_telehealth": {
        "policy_family": "telehealth_reimbursement",
        "version": 1,
        "effective_date": "2026-01-01",
        "is_current": True,
    },
}

# Attach metadata to each doc
for doc in SAMPLE_DOCS:
    doc["metadata"] = DOC_METADATA.get(doc["id"], {
        "policy_family": doc["id"], "version": 1,
        "effective_date": "unknown", "is_current": True,
    })

print("Metadata attached. Current documents:")
for doc in SAMPLE_DOCS:
    if doc["metadata"]["is_current"]:
        print(f"  - {doc['title']}")

## Step 2 — Detect changes between versions

For any policy family with more than one version, compare the old and new text and ask the
LLM to summarize exactly what changed. This is the core "diff" feature.

In [ ]:
def find_version_pairs():
    """Groups documents by policy family and returns (old, new) pairs where >1 version exists."""
    families = {}
    for doc in SAMPLE_DOCS:
        fam = doc["metadata"]["policy_family"]
        families.setdefault(fam, []).append(doc)

    pairs = []
    for fam, docs in families.items():
        if len(docs) > 1:
            docs_sorted = sorted(docs, key=lambda d: d["metadata"]["version"])
            pairs.append((docs_sorted[-2], docs_sorted[-1]))  # previous vs current
    return pairs

pairs = find_version_pairs()
print(f"Found {len(pairs)} policy update(s) to analyze.")
for old, new in pairs:
    print(f"  - {old['title']}  -->  {new['title']}")

In [ ]:
def detect_changes(old_doc, new_doc, max_new_tokens=250):
    """Asks the LLM to summarize exactly what changed between two versions of a policy."""
    prompt = f"""You are a Medicaid compliance analyst. Compare these two versions of the
same policy and list ONLY the concrete changes (numbers, deadlines, thresholds, wording that
changes an obligation). Be specific and use a short bullet list. If nothing substantive
changed, say so.

OLD VERSION ({old_doc['metadata']['effective_date']}):
{old_doc['text']}

NEW VERSION ({new_doc['metadata']['effective_date']}):
{new_doc['text']}

Changes:"""

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=0.2, do_sample=True,
        )

    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


# Run change detection on every version pair found
change_reports = []
for old, new in pairs:
    print(f"Analyzing: {new['title']}...")
    changes = detect_changes(old, new)
    change_reports.append({
        "policy_family": new["metadata"]["policy_family"],
        "old_doc": old["title"],
        "new_doc": new["title"],
        "changes": changes,
    })
    print(changes)
    print("-" * 60)

## Step 3 — Map changes to affected internal SOPs

In a real deployment, a client would upload their internal SOPs/procedures. Here we simulate
that with a small sample set of "internal procedures" and have the model flag which ones are
affected by each detected change.

In [ ]:
# Simulated internal client SOPs (in a real deployment these come from the client's own docs)
SAMPLE_INTERNAL_SOPS = [
    {"id": "sop_001", "title": "Outpatient PT Intake Procedure",
     "text": "Front desk staff verify visit count against the 12-visit prior authorization "
             "threshold before scheduling. If a patient exceeds 12 visits, flag for prior auth."},
    {"id": "sop_002", "title": "Claims Denial Appeals Process",
     "text": "Appeals for denied physical therapy claims must reference the prior authorization "
             "policy visit threshold and the 5-business-day submission window."},
    {"id": "sop_003", "title": "Telehealth Billing Checklist",
     "text": "Confirm modality (audio-only vs video) is documented on every telehealth claim "
             "before submission."},
]

def find_affected_sops(change_summary, sops, max_new_tokens=200):
    """Asks the LLM which internal SOPs are affected by a given policy change."""
    sop_list = "\n\n".join(f"[{s['id']}] {s['title']}: {s['text']}" for s in sops)
    prompt = f"""Given this policy change:
{change_summary}

And these internal procedures:
{sop_list}

Which procedure(s) are affected by this change, and why? List by ID. If none are affected, say so."""

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens, temperature=0.2, do_sample=True,
        )
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)


# Run impact mapping for each detected change
for report in change_reports:
    print(f"=== Impact analysis: {report['new_doc']} ===")
    impact = find_affected_sops(report["changes"], SAMPLE_INTERNAL_SOPS)
    report["affected_sops"] = impact
    print(impact)
    print("-" * 60)

## Step 4 — Generate the full compliance report

This ties Phase 3 together: change detected → impact mapped → a clean, structured report a
compliance analyst could actually act on.

In [ ]:
def generate_compliance_report(report):
    """Formats a single change report into a clean, readable summary."""
    output = f"""
{'=' * 70}
COMPLIANCE ALERT — {report['policy_family']}
{'=' * 70}

Previous version: {report['old_doc']}
Current version:  {report['new_doc']}

WHAT CHANGED:
{report['changes']}

AFFECTED INTERNAL PROCEDURES:
{report['affected_sops']}

STATUS: Pending review — draft SOP updates not yet generated
{'=' * 70}
"""
    return output

for report in change_reports:
    print(generate_compliance_report(report))

---
## What Phase 3 Adds

Your engine now:
- Knows which policy version is current vs outdated
- Automatically detects and summarizes what changed between versions
- Maps changes to specific affected internal procedures
- Produces a structured compliance report, not just a raw answer

This is the demo-ready core: point it at real public CMS policy updates and a sample set of
internal SOPs, and it produces exactly the "detect impact → draft update" flow we scoped for
your agency/investor demo.

## Next: Phase 4 — Draft Generation
The next step is generating an actual **draft SOP update** (not just flagging what's affected),
with citations back to the source policy — the piece that makes this a finished demo flow.